In [ ]:
# ==============================================================================
# Cell 1: Clean Environment Setup & Verify In-Graph ConvSTFT + Custom tr/ Dataset
# ==============================================================================
!pip install -q huggingface_hub pyarrow onnx onnxruntime onnxscript librosa soundfile
%cd /kaggle/working
!rm -rf trialSpeakSafe
!git clone https://github.com/rayafa2005/trialSpeakSafe.git
%cd /kaggle/working/trialSpeakSafe

import os, glob, torch
from model.tatvanet import TatvaNet
from features.spectrogram import _to_db, TOP_DB

# 1. Verify top_db=80.0 dynamic range clamping is active inside ConvSTFT
test_spec = torch.tensor([[[1e-5, 1.0, 10.0]]])
db_out = _to_db(test_spec, top_db=80.0)
db_span = float(db_out.max() - db_out.min())
assert db_span <= 80.01, f"top_db clamping not active! Span={db_span}"
print(f"[OK] In-graph ConvSTFT _to_db(top_db={TOP_DB}) verified: max-min span = {db_span:.2f} dB")

# 2. Verify custom tr/ real dataset and test_audio(razz) benchmark are present
tr_files = sorted(glob.glob("tr/*"))
razz_files = sorted(glob.glob("test_audio(razz)/fake/*"))
print(f"[OK] Found {len(tr_files)} real mobile recordings in tr/ and {len(razz_files)} benchmark files in test_audio(razz)/fake/")

In [ ]:
# ==============================================================================
# Cell 2: Process Custom tr/ Real Dataset (16 Speakers) + Download External Data
# ==============================================================================
import os, io, glob, random, subprocess, shutil
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from huggingface_hub import HfApi, hf_hub_download, login
import pyarrow.parquet as pq

# Optional HuggingFace token login (reads from Kaggle Secrets if configured)
HF_TOKEN = os.environ.get("HF_TOKEN", None)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if HF_TOKEN:
    login(token=HF_TOKEN)

SAMPLE_RATE = 16000
CHUNK_SAMPLES = 32000  # 2.0s @ 16kHz

def decode_any_bytes_or_file(src_path_or_bytes, is_bytes=False):
    """Decodes any audio file or raw parquet bytes into 16kHz mono float32 numpy array."""
    if not is_bytes:
        cmd = [
            "ffmpeg", "-v", "error", "-i", str(src_path_or_bytes),
            "-f", "f32le", "-acodec", "pcm_f32le", "-ac", "1", "-ar", str(SAMPLE_RATE), "pipe:1"
        ]
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if res.returncode == 0 and len(res.stdout) >= 4:
            return np.frombuffer(res.stdout, dtype=np.float32).copy()
    else:
        try:
            data, sr = sf.read(io.BytesIO(src_path_or_bytes), dtype="float32")
            if data.ndim > 1:
                data = data.mean(axis=1)
            if sr != SAMPLE_RATE:
                t = torch.from_numpy(data).unsqueeze(0)
                data = torchaudio.functional.resample(t, sr, SAMPLE_RATE).squeeze(0).numpy()
            return data.astype(np.float32)
        except Exception:
            cmd = [
                "ffmpeg", "-v", "error", "-i", "pipe:0",
                "-f", "f32le", "-acodec", "pcm_f32le", "-ac", "1", "-ar", str(SAMPLE_RATE), "pipe:1"
            ]
            res = subprocess.run(cmd, input=src_path_or_bytes, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            if res.returncode == 0 and len(res.stdout) >= 4:
                return np.frombuffer(res.stdout, dtype=np.float32).copy()
    return None

def strip_silence_np(wav, frame_len=160, thresh_ratio=0.05):
    """Strips leading and trailing silence using short-time frame RMS."""
    wav = wav - np.mean(wav)
    n = len(wav)
    if n < frame_len * 6:
        return wav
    n_frames = n // frame_len
    frames = wav[:n_frames * frame_len].reshape(n_frames, frame_len)
    rms = np.sqrt(np.mean(frames ** 2, axis=1))
    peak_rms = np.max(rms)
    if peak_rms > 1e-6:
        active = np.where(rms > peak_rms * thresh_ratio)[0]
        if len(active) > 0:
            s = max(0, int(active[0]) * frame_len - frame_len)
            e = min(n, (int(active[-1]) + 2) * frame_len)
            if (e - s) >= 8000:
                return wav[s:e]
    return wav

# ------------------------------------------------------------------------------
# PART A: Process Custom tr/ Real Mobile Recordings (46 files, 16 speakers)
# Strictly split by speaker FIRST (13 Train speakers, 3 Val speakers)
# ------------------------------------------------------------------------------
print("\n=== Part A: Processing Custom tr/ Real Mobile Dataset ===")
os.makedirs("data/raw/tr_chunks/train", exist_ok=True)
os.makedirs("data/raw/tr_chunks/val", exist_ok=True)

tr_files = sorted([f for f in glob.glob("tr/*") if os.path.isfile(f)])
speaker_to_files = {}
for fpath in tr_files:
    base = os.path.splitext(os.path.basename(fpath))[0]
    spk = base.rsplit(" - ", 1)[-1].strip().lower() if " - " in base else base.lower()
    spk_id = "tr_spk_" + "".join(c if c.isalnum() else "_" for c in spk)
    speaker_to_files.setdefault(spk_id, []).append(fpath)

all_tr_speakers = sorted(speaker_to_files.keys())
rng = random.Random(42)
rng.shuffle(all_tr_speakers)
# Hold out 3 speakers strictly for Validation, 13 speakers for Training
tr_val_speakers = set(all_tr_speakers[:3])
tr_train_speakers = set(all_tr_speakers[3:])
assert len(tr_train_speakers & tr_val_speakers) == 0, "Speaker leakage in tr/!"

tr_records = []
for spk_id, fpaths in speaker_to_files.items():
    split_name = "val" if spk_id in tr_val_speakers else "train"
    stride = 32000 if split_name == "val" else 8000  # 0.5s stride in train, non-overlapping 2.0s in val
    for f_idx, fpath in enumerate(fpaths):
        wav = decode_any_bytes_or_file(fpath, is_bytes=False)
        if wav is None or len(wav) < 8000:
            continue
        wav = strip_silence_np(wav)
        if len(wav) <= CHUNK_SAMPLES:
            chunk = np.pad(wav, (0, CHUNK_SAMPLES - len(wav)))
            peak = np.max(np.abs(chunk))
            if peak > 1e-6:
                chunk = chunk / peak
            out_wav = f"data/raw/tr_chunks/{split_name}/{spk_id}_f{f_idx}_c0.wav"
            sf.write(out_wav, chunk, SAMPLE_RATE)
            tr_records.append({"path": os.path.abspath(out_wav), "label": 0, "speaker_id": spk_id, "split": split_name, "domain": "tr_real"})
        else:
            c_idx = 0
            for start in range(0, len(wav) - CHUNK_SAMPLES + 1, stride):
                chunk = wav[start:start + CHUNK_SAMPLES].copy()
                if np.sqrt(np.mean(chunk ** 2)) < 1e-4:
                    continue
                peak = np.max(np.abs(chunk))
                if peak > 1e-6:
                    chunk = chunk / peak
                out_wav = f"data/raw/tr_chunks/{split_name}/{spk_id}_f{f_idx}_c{c_idx}.wav"
                sf.write(out_wav, chunk, SAMPLE_RATE)
                tr_records.append({"path": os.path.abspath(out_wav), "label": 0, "speaker_id": spk_id, "split": split_name, "domain": "tr_real"})
                c_idx += 1

df_tr = pd.DataFrame(tr_records)
df_tr.to_csv("data/raw/tr_chunks/tr_metadata.csv", index=False)
n_tr_train = int((df_tr["split"] == "train").sum()) if len(df_tr) > 0 else 0
n_tr_val = int((df_tr["split"] == "val").sum()) if len(df_tr) > 0 else 0
print(f"[OK] Custom tr/ chunks saved: {len(df_tr)} total (Train={n_tr_train} across {len(tr_train_speakers)} speakers, Val={n_tr_val} across {len(tr_val_speakers)} speakers)")

# ------------------------------------------------------------------------------
# PART B: Fetch Real Indian Voices (ai4bharat/indicvoices_r) with True Speaker ID
# ------------------------------------------------------------------------------
print("\n=== Part B: Fetching Real Indian Voices (ai4bharat/indicvoices_r) ===")
os.makedirs("data/raw/indic_voices", exist_ok=True)
api = HfApi()
REPO_INDIC = "ai4bharat/indicvoices_r"
LANGUAGES = {"hindi": 800, "kannada": 500, "telugu": 500, "kashmiri": 300, "tamil": 400, "bengali": 400}
indic_records = []

try:
    all_indic_files = api.list_repo_files(repo_id=REPO_INDIC, repo_type="dataset")
    for lang, target_cnt in LANGUAGES.items():
        parquets = [f for f in all_indic_files if lang in f.lower() and f.endswith(".parquet")]
        saved = 0
        for pfile in parquets:
            if saved >= target_cnt:
                break
            try:
                local_pq = hf_hub_download(repo_id=REPO_INDIC, filename=pfile, repo_type="dataset")
                df_pq = pq.read_table(local_pq).to_pandas()
                for idx, row in df_pq.iterrows():
                    if saved >= target_cnt:
                        break
                    aud = row.get("audio")
                    raw_bytes = aud.get("bytes") if isinstance(aud, dict) else None
                    if not raw_bytes:
                        continue
                    wav = decode_any_bytes_or_file(raw_bytes, is_bytes=True)
                    if wav is None or len(wav) < 12000:
                        continue
                    wav = strip_silence_np(wav)
                    raw_spk = str(row.get("speaker_id") or row.get("ULCA_speaker_id") or f"shard_{idx // 10}")
                    spk_id = f"indic_{lang}_{raw_spk}"
                    out_wav = f"data/raw/indic_voices/indic_{lang}_{saved:04d}.wav"
                    sf.write(out_wav, wav[:SAMPLE_RATE * 6], SAMPLE_RATE)
                    indic_records.append({"path": os.path.abspath(out_wav), "label": 0, "speaker_id": spk_id, "domain": "indic_real"})
                    saved += 1
                os.remove(local_pq)
            except Exception as e:
                print(f"  [Skip shard {pfile}]: {e}")
        print(f"  -> {lang}: saved {saved} decoded 16kHz WAVs")
except Exception as e:
    print(f"[IndicVoices Warning]: {e}")

df_indic = pd.DataFrame(indic_records)
df_indic.to_csv("data/raw/indic_voices/indic_metadata.csv", index=False)
print(f"[OK] Total IndicVoices saved: {len(df_indic)}")

# ------------------------------------------------------------------------------
# PART C: Fetch Verified Modern Neural TTS / ElevenLabs / VoxGuard Fakes
# ------------------------------------------------------------------------------
print("\n=== Part C: Fetching Verified Modern Synthetic AI Voices ===")
os.makedirs("data/raw/modern_fakes", exist_ok=True)
MODERN_REPOS = [
    "skypro1111/elevenlabs_dataset",
    "gereon/voxguard-synthetic-speech",
    "garystafford/deepfake-audio-detection",
]
modern_records = []
TARGET_MODERN = 2500

for repo in MODERN_REPOS:
    if len(modern_records) >= TARGET_MODERN:
        break
    try:
        rfiles = api.list_repo_files(repo_id=repo, repo_type="dataset")
        pfiles = [f for f in rfiles if f.endswith(".parquet")][:4]
        for pf in pfiles:
            if len(modern_records) >= TARGET_MODERN:
                break
            local_pq = hf_hub_download(repo_id=repo, filename=pf, repo_type="dataset")
            df_pq = pq.read_table(local_pq).to_pandas()
            cols_lower = {c.lower(): c for c in df_pq.columns}
            lbl_col = cols_lower.get("label") or cols_lower.get("is_fake") or cols_lower.get("target") or cols_lower.get("class")
            spk_col = cols_lower.get("speaker_id") or cols_lower.get("voice_id") or cols_lower.get("speaker") or cols_lower.get("model")

            for idx, row in df_pq.iterrows():
                if len(modern_records) >= TARGET_MODERN:
                    break
                if lbl_col is not None:
                    val_str = str(row[lbl_col]).strip().lower()
                    if val_str in {"real", "bonafide", "human", "true_human", "false", "0"}:
                        continue
                aud = row.get("audio")
                raw_bytes = aud.get("bytes") if isinstance(aud, dict) else None
                if not raw_bytes:
                    continue
                wav = decode_any_bytes_or_file(raw_bytes, is_bytes=True)
                if wav is None or len(wav) < 12000:
                    continue
                wav = strip_silence_np(wav)
                raw_spk = str(row[spk_col]) if (spk_col and pd.notna(row[spk_col])) else f"grp_{len(modern_records) // 12}"
                repo_tag = repo.split("/")[0]
                spk_id = f"modfake_{repo_tag}_{raw_spk}"
                out_wav = f"data/raw/modern_fakes/modern_fake_{len(modern_records):04d}.wav"
                sf.write(out_wav, wav[:SAMPLE_RATE * 6], SAMPLE_RATE)
                modern_records.append({"path": os.path.abspath(out_wav), "label": 1, "speaker_id": spk_id, "domain": "modern_fake"})
            os.remove(local_pq)
        print(f"  -> After {repo}: {len(modern_records)} verified modern fake clips")
    except Exception as e:
        print(f"  [Skip {repo}]: {e}")

df_modern = pd.DataFrame(modern_records)
df_modern.to_csv("data/raw/modern_fakes/modern_metadata.csv", index=False)
print(f"[OK] Total Modern Synthetic AI clips saved: {len(df_modern)}")

In [ ]:
# ==============================================================================
# Cell 3: Zero-Leakage, Speaker-Disjoint Master Manifest Builder (Fast String Mode)
# ==============================================================================
import os, glob, random
import pandas as pd

random.seed(42)
AUDIO_EXTS = {".flac", ".wav", ".mp3", ".m4a", ".ogg", ".opus"}
kaggle_folders = sorted(glob.glob("/kaggle/input/*"))
print("Scanning /kaggle/input folders:", kaggle_folders, flush=True)

# 1. Parse ALL ASVspoof CM protocol files
protocol_map = {}
for kdir in kaggle_folders:
    for root, _, files in os.walk(kdir):
        for f in files:
            if f.endswith(".txt") and any(k in f.lower() for k in ["cm", "protocol", "trn", "trl"]):
                p_path = os.path.join(root, f)
                try:
                    with open(p_path, "r", encoding="utf-8", errors="ignore") as pf:
                        for line in pf:
                            parts = line.strip().split()
                            if len(parts) >= 5 and parts[-1].lower() in ("bonafide", "spoof"):
                                spk_id = f"asv_{parts[0]}"
                                file_key = parts[1]
                                lbl = 0 if parts[-1].lower() == "bonafide" else 1
                                protocol_map[file_key] = (lbl, spk_id)
                except Exception:
                    pass
print(f"[OK] Loaded {len(protocol_map):,} protocol entries from all ASVspoof splits.", flush=True)

# 2. Fast single-pass scan of /kaggle/input without slow realpath/relpath FUSE syscalls
kaggle_real, kaggle_fake = [], []
seen_names = set()
total_scanned = 0

# Limit to 20,000 per class from input (we only need ~8,000 to balance with tr/ and IndicVoices)
MAX_INPUT_REAL = 20000
MAX_INPUT_FAKE = 20000

for kdir in kaggle_folders:
    kdir_str = str(kdir).replace("\\", "/")
    for root, _, files in os.walk(kdir):
        root_clean = root.replace("\\", "/")
        rel_folder = root_clean[len(kdir_str):].strip("/")
        parent_grp = rel_folder.replace("/", "_") if rel_folder else "root"

        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext not in AUDIO_EXTS:
                continue

            fname_no_ext = os.path.splitext(f)[0]
            if fname_no_ext in seen_names:
                continue
            seen_names.add(fname_no_ext)
            total_scanned += 1

            if total_scanned % 30000 == 0:
                print(f"  Scanned {total_scanned:,} audio files in input (Real: {len(kaggle_real):,}, Fake: {len(kaggle_fake):,})...", flush=True)

            # Fast string concatenation (0 disk/FUSE syscalls!)
            full_path = f"{root_clean}/{f}"
            lower_path = full_path.lower()

            if fname_no_ext in protocol_map:
                lbl, spk_id = protocol_map[fname_no_ext]
                rec = {"path": full_path, "label": lbl, "speaker_id": spk_id, "domain": "asvspoof"}
                if lbl == 0:
                    if len(kaggle_real) < MAX_INPUT_REAL:
                        kaggle_real.append(rec)
                else:
                    if len(kaggle_fake) < MAX_INPUT_FAKE:
                        kaggle_fake.append(rec)
            else:
                if any(k in lower_path for k in ["elevenlabs", "wavefake", "fake", "deepfake", "synthetic", "cloned", "tts", "spoof"]):
                    if len(kaggle_fake) < MAX_INPUT_FAKE:
                        spk_id = f"kag_fake_{parent_grp}_{fname_no_ext[:6]}"
                        kaggle_fake.append({"path": full_path, "label": 1, "speaker_id": spk_id, "domain": "kaggle_fake"})
                elif any(k in lower_path for k in ["libri", "clean", "human", "real", "bonafide", "common_voice"]):
                    if len(kaggle_real) < MAX_INPUT_REAL:
                        tok = fname_no_ext.split("-")[0] if "-" in fname_no_ext else parent_grp
                        spk_id = f"kag_real_{tok}"
                        kaggle_real.append({"path": full_path, "label": 0, "speaker_id": spk_id, "domain": "kaggle_real"})

print(f"[OK] Total scanned: {total_scanned:,} | Matched {len(kaggle_real):,} Real | {len(kaggle_fake):,} Fake", flush=True)

# 3. Load Part B (IndicVoices) & Part C (Modern Fakes) & Part A (Custom tr/) metadata
indic_recs = pd.read_csv("data/raw/indic_voices/indic_metadata.csv").to_dict("records") if os.path.exists("data/raw/indic_voices/indic_metadata.csv") else []
modern_recs = pd.read_csv("data/raw/modern_fakes/modern_metadata.csv").to_dict("records") if os.path.exists("data/raw/modern_fakes/modern_metadata.csv") else []
tr_df = pd.read_csv("data/raw/tr_chunks/tr_metadata.csv") if os.path.exists("data/raw/tr_chunks/tr_metadata.csv") else pd.DataFrame()

def split_domain_by_speaker(records, max_total=12000, seed=42):
    """Splits records strictly by speaker_id into (train, val, test) without speaker leakage."""
    if not records:
        return [], [], []
    rng = random.Random(seed)
    by_spk = {}
    for r in records:
        by_spk.setdefault(str(r["speaker_id"]), []).append(r)
    spks = sorted(by_spk.keys())
    rng.shuffle(spks)
    n = len(spks)
    n_tr = max(1, int(0.80 * n)) if n >= 3 else n
    n_va = max(1, int(0.10 * n)) if n >= 3 else 0
    tr_spks = set(spks[:n_tr])
    va_spks = set(spks[n_tr:n_tr + n_va])
    te_spks = set(spks[n_tr + n_va:])

    tr = [r for s in tr_spks for r in by_spk[s]]
    va = [r for s in va_spks for r in by_spk[s]]
    te = [r for s in te_spks for r in by_spk[s]]
    rng.shuffle(tr); rng.shuffle(va); rng.shuffle(te)
    return tr[:int(max_total * 0.8)], va[:int(max_total * 0.1)], te[:int(max_total * 0.1)]

# Split each source domain by speaker independently
asv_real = [r for r in kaggle_real if r["domain"] == "asvspoof"]
oth_real = [r for r in kaggle_real if r["domain"] != "asvspoof"]
asv_fake = [r for r in kaggle_fake if r["domain"] == "asvspoof"]
oth_fake = [r for r in kaggle_fake if r["domain"] != "asvspoof"]

tr_asv_r, va_asv_r, te_asv_r = split_domain_by_speaker(asv_real, max_total=5000, seed=42)
tr_oth_r, va_oth_r, te_oth_r = split_domain_by_speaker(oth_real, max_total=4000, seed=43)
tr_ind_r, va_ind_r, te_ind_r = split_domain_by_speaker(indic_recs, max_total=4000, seed=44)

tr_asv_f, va_asv_f, te_asv_f = split_domain_by_speaker(asv_fake, max_total=6000, seed=45)
tr_oth_f, va_oth_f, te_oth_f = split_domain_by_speaker(oth_fake, max_total=5000, seed=46)
tr_mod_f, va_mod_f, te_mod_f = split_domain_by_speaker(modern_recs, max_total=4000, seed=47)

# Add custom tr/ chunks (oversampled 3x ONLY inside Train so model strongly learns phone-mic/WhatsApp real speech; 1x in Val)
tr_custom_train = tr_df[tr_df["split"] == "train"].to_dict("records") if len(tr_df) > 0 else []
tr_custom_val   = tr_df[tr_df["split"] == "val"].to_dict("records")   if len(tr_df) > 0 else []

real_train = tr_asv_r + tr_oth_r + tr_ind_r + (tr_custom_train * 3)
real_val   = va_asv_r + va_oth_r + va_ind_r + tr_custom_val
real_test  = te_asv_r + te_oth_r + te_ind_r

fake_train = tr_asv_f + tr_oth_f + tr_mod_f
fake_val   = va_asv_f + va_oth_f + va_mod_f
fake_test  = te_asv_f + te_oth_f + te_mod_f

# Verify ZERO speaker overlap and ZERO file overlap across splits BEFORE balancing/shuffling
train_spks = {r["speaker_id"] for r in real_train + fake_train}
val_spks   = {r["speaker_id"] for r in real_val + fake_val}
test_spks  = {r["speaker_id"] for r in real_test + fake_test}
assert len(train_spks & val_spks) == 0, f"Speaker overlap between train and val: {train_spks & val_spks}"
assert len(train_spks & test_spks) == 0, f"Speaker overlap between train and test: {train_spks & test_spks}"
assert len(val_spks & test_spks) == 0, f"Speaker overlap between val and test: {val_spks & test_spks}"

train_paths = {r["path"] for r in real_train + fake_train}
val_paths   = {r["path"] for r in real_val + fake_val}
test_paths  = {r["path"] for r in real_test + fake_test}
assert len(train_paths & val_paths) == 0, "File leakage between train and val!"
assert len(train_paths & test_paths) == 0, "File leakage between train and test!"
print("[OK] ZERO-LEAKAGE VERIFIED: 0 overlapping speakers and 0 overlapping files across Train, Val, and Test!", flush=True)

def balance_and_shuffle_within_split(reals, fakes, seed=42):
    """Balances 1:1 and shuffles strictly inside a single split."""
    rng = random.Random(seed)
    r_copy = list(reals)
    f_copy = list(fakes)
    rng.shuffle(r_copy)
    rng.shuffle(f_copy)
    m = min(len(r_copy), len(f_copy))
    combined = r_copy[:m] + f_copy[:m]
    rng.shuffle(combined)
    return pd.DataFrame(combined)

df_train = balance_and_shuffle_within_split(real_train, fake_train, seed=101)
df_val   = balance_and_shuffle_within_split(real_val,   fake_val,   seed=102)
df_test  = balance_and_shuffle_within_split(real_test,  fake_test,  seed=103)

os.makedirs("data", exist_ok=True)
df_train.to_csv("data/train_manifest_kaggle.csv", index=False)
df_val.to_csv("data/val_manifest_kaggle.csv", index=False)
df_test.to_csv("data/test_manifest_kaggle.csv", index=False)

print(f"  -> Train: {len(df_train):,} samples (50% Real / 50% Fake)", flush=True)
print(f"  -> Val:   {len(df_val):,} samples (50% Real / 50% Fake)", flush=True)
print(f"  -> Test:  {len(df_test):,} samples (50% Real / 50% Fake)", flush=True)

In [ ]:
# ==============================================================================
# Cell 4: Train TatvaNet From Scratch (Live Streaming Output)
# ==============================================================================
import subprocess, sys

cmd = [
    sys.executable, "-u", "training/train.py",
    "--train_manifest", "data/train_manifest_kaggle.csv",
    "--val_manifest", "data/val_manifest_kaggle.csv",
    "--epochs", "30",
    "--batch_size", "64",
    "--lr", "0.0005",
    "--num_workers", "4",
    "--checkpoint_dir", "checkpoints",
    "--reset_best"
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
assert proc.returncode == 0, f"Training failed with exit code {proc.returncode}"

In [ ]:
# ==============================================================================
# Cell 5 (Optional): Fine-Tune with Baseline EER Protection Gate
# ==============================================================================
import subprocess, sys

cmd = [
    sys.executable, "-u", "training/train.py",
    "--train_manifest", "data/train_manifest_kaggle.csv",
    "--val_manifest", "data/val_manifest_kaggle.csv",
    "--finetune", "checkpoints/best_model.pt",
    "--epochs", "8",
    "--batch_size", "64",
    "--lr", "0.0001",
    "--num_workers", "4",
    "--checkpoint_dir", "checkpoints"
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
assert proc.returncode == 0, f"Fine-tuning failed with exit code {proc.returncode}"

In [ ]:
# ==============================================================================
# Cell 6: Export ONNX to /kaggle/working/tatvanet.onnx & Verify Parity
# ==============================================================================
!pip install -q onnxscript
import shutil, torch, numpy as np, onnxruntime as ort
from model.tatvanet import TatvaNet
from export.export_tflite import export_onnx

device = torch.device("cpu")
ckpt_path = "checkpoints/best_model.pt"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
ep = ckpt.get("epoch", 0) + 1
best_e = 100.0 * ckpt.get("best_eer", 0.0)
val_e  = 100.0 * ckpt.get("val_eer", 0.0)
print(f"Loaded {ckpt_path}: Epoch={ep}, Best Val EER={best_e:.2f}% (Val EER={val_e:.2f}%)", flush=True)

model = TatvaNet(sample_rate=16000, n_classes=2).to(device)
model.load_state_dict(ckpt["model_state"], strict=True)
model.eval()
print("[OK] strict=True state_dict load succeeded with 0 missing/unexpected keys!", flush=True)

# Export to /kaggle/working/tatvanet.onnx (stays on Kaggle until you download it)
onnx_out = "/kaggle/working/tatvanet.onnx"
pt_out   = "/kaggle/working/best_model.pt"
export_onnx(model, onnx_out, clip_samples=32000, device=device)
shutil.copy(ckpt_path, pt_out)

# Verify numerical parity between PyTorch and ONNX Runtime
sess = ort.InferenceSession(onnx_out, providers=["CPUExecutionProvider"])
dummy = torch.randn(2, 32000)
with torch.no_grad():
    pt_logits = model(dummy)["logits"].numpy()
ort_logits = sess.run(None, {"audio": dummy.numpy()})[0]
max_diff = float(np.max(np.abs(pt_logits - ort_logits)))
print(f"[OK] PyTorch vs ONNX max logit difference: {max_diff:.6f}", flush=True)
assert max_diff < 1e-3, f"ONNX parity drift too high: {max_diff}"

In [ ]:
# ==============================================================================
# Cell 7: Gatekeeper Benchmark on test_audio(razz) + Custom tr/ & Download Links
# ==============================================================================
import subprocess, sys
cmd = [sys.executable, "-u", "TEST_FOLDER_AUDIOS.py", "--onnx", "/kaggle/working/tatvanet.onnx", "--checkpoint", "/kaggle/working/best_model.pt"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

from IPython.display import FileLink, display
print("\n=== 1-Click Download Links (Download after verifying benchmark above) ===", flush=True)
%cd /kaggle/working
display(FileLink("tatvanet.onnx"))
display(FileLink("best_model.pt"))
%cd /kaggle/working/trialSpeakSafe